In [8]:
import kafi.streams.topologynode
from kafi.streams.topologynode import source

#

order_source_str = "orders"
customer_source_str = "customers"
#
order_source_tn = source(order_source_str)
customer_source_tn = source(customer_source_str)
#
root_tn = (
    order_source_tn
    .join_equi(customer_source_tn,
               lambda l: l["customer_id"],
               lambda r: r["customer_id"],
               lambda l, r: {"order_id": l["order_id"], "customer_id": l["customer_id"], "name": r["name"]})
)
#
root_tn.build()


In [9]:
def gen_order(order_id_int, customer_id_int, ts_int):
    return {"order_id": order_id_int,
            "customer_id": customer_id_int,
            "ts": ts_int}

def gen_customer(customer_id_int, name_str):
    return {"customer_id": customer_id_int,
            "name": name_str}

#

root_tn.push(order_source_str, [gen_order(1, 1, 0)])
root_tn.push(customer_source_str, [gen_customer(1, "Bea")])
root_tn.latest()

[{'order_id': 1, 'customer_id': 1, 'name': 'Bea'}]

In [ ]:
import kafi.streams.topologynode
from kafi.streams.topologynode import source

#

order_source_str = "orders"
customer_source_str = "customers"
clock_source_str = "clock"
#
order_source_tn = source(order_source_str)
customer_source_tn = source(customer_source_str)
#
order_window_tn = (
    order_source_tn
    .map(lambda x: x | {"window_start": x["ts"], "window_end": x["ts"] + 10})
)
order_windowing_tn = (
    order_window_tn
    .join_equi(order_window_tn,
               lambda l: l["window_end"],
               lambda r: r["window_start"],
               lambda l, _: l)
    .neg()
    ._peek()
    .union(order_window_tn)
    .distinct()
)
#
root_tn = (
    order_windowing_tn
    .join_equi(customer_source_tn,
               lambda l: l["customer_id"],
               lambda r: r["customer_id"],
               lambda l, r: {"order_id": l["order_id"], "customer_id": l["customer_id"], "name": r["name"],
                             "window_start": l["window_start"], "window_end": l["window_end"]})
)
#
root_tn.build()
root_tn._from_zSet_function = root_tn.zSet_to_record_any_weight_int_tuple_list



In [61]:
root_tn.push(order_source_str, [gen_order(1, 1, 0)])
root_tn.push(customer_source_str, [gen_customer(1, "Peter")])
root_tn.latest()


[({'order_id': 1,
   'customer_id': 1,
   'name': 'Peter',
   'window_start': 0,
   'window_end': 10},
  1)]

In [62]:
root_tn.push(order_source_str, [gen_order(2, 2, 10)])
root_tn.latest()


({'order_id': 1, 'customer_id': 1, 'ts': 0, 'window_start': 0, 'window_end': 10}, -1)


[]

In [63]:
import importlib
importlib.reload(kafi.streams.topologynode)

<module 'kafi.streams.topologynode' from '/home/ralph/github/kafi/kafi/streams/topologynode.py'>